# Visual Analytics

## Assignment 3

**Instructor:** Dr. Marco D'Ambros  
**TAs:** Giuseppe Crupi, Mattia Giannaccari

**Contacts:** marco.dambros@usi.ch, giuseppe.crupi@usi.ch, mattia.giannaccari@usi.ch

**Due Date:** May 25, 2026 @ 23:55

---
The goal of this assignment is to use **Spark (PySpark)** and **Polars** in Jupyter notebooks.  
The files `trip_data.csv`, `trip_fare.csv`, and `nyc_boroughs.geojson` are available in the provided folder: [Assignment3-data](https://usi365-my.sharepoint.com/:f:/g/personal/armenc_usi_ch/Ejp7sb8QAMROoWe0XUDcAkMBoqUFk-w2Vgroup025NhAww?e=2I7SMC).

- Use **Spark** to solve **Exercises 1–4**
- Use **Polars** to solve **Exercises 5–8**

Please name your notebook file as `SurnameName_Assignment3.ipynb`

# ⚡️ Spark Exercises (50 pts)

### Importing the libraries

In [1]:
import os
import json

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql import Window

from shapely.geometry import shape, Point

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, FactorRange, Legend, LegendItem
from bokeh.transform import factor_cmap
from bokeh.palettes import Spectral4

### Initial Spark Setup

In [2]:
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = "python"

spark = SparkSession.builder \
    .appName("Assignment3") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.range(5).show()

26/05/26 09:57:21 WARN Utils: Your hostname, USILU-6274.local resolves to a loopback address: 127.0.0.1; using 10.21.49.104 instead (on interface en0)
26/05/26 09:57:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/26 09:57:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



### Importing and cleaning the data

In [3]:
# Loading the datasets
# After selecting the dataset thanks to the related paths, header=True is used to consider the first row as column names while 
# inferSchema=True is used to automatically detect data types

trip_data = spark.read.csv(
    "data-assignment3/trip_data.csv", header=True, inferSchema=True
)

trip_fare = spark.read.csv(
    "data-assignment3/trip_fare.csv", header=True, inferSchema=True
)

# Since the column names have spaces, they need to be trimmed before joining
trip_data = trip_data.toDF(*[c.strip() for c in trip_data.columns])
trip_fare = trip_fare.toDF(*[c.strip() for c in trip_fare.columns])

# Visualizing the clean datasets without spaces in column names
trip_data.show(5)
trip_fare.show(5)

+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|           medallion|        hack_license|vendor_id|rate_code|store_and_fwd_flag|    pickup_datetime|   dropoff_datetime|passenger_count|trip_time_in_secs|trip_distance|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|
+--------------------+--------------------+---------+---------+------------------+-------------------+-------------------+---------------+-----------------+-------------+----------------+---------------+-----------------+----------------+
|89D227B655E5C82AE...|BA96DE419E711691B...|      CMT|        1|                 N|2013-01-01 15:11:48|2013-01-01 15:18:10|              4|              382|          1.0|      -73.978165|      40.757977|       -73.989838|       40.751171|
|0BD7C8F5BA12B88E0...|9FD8F69F0804BDB55...| 

### Exercise 1 (8 pts)
Join the `trip_data` and `trip_fare` dataframes into one, considering only trips from January 1–7, 2013. Filter out trips where the total amount charged is 0 or less, or the trip distance is 0 or less. Report how many rows are removed by each filter, the total number of rows after filtering, and the average tip amount across the resulting dataset.

### Solution 1

#### Merging the two dataframes

In [4]:
# Merging the datasets on the selected columns: medallion, hack_license, pickup_datetime
# This is because the medallion and hack_license columns identify the taxi and the driver, 
# while the pickup_datetime column identifies the specific trip.
df = trip_data.join(
    trip_fare,
    on=["medallion", "hack_license", "pickup_datetime"],
    how="inner"
)

print(f"Rows after merging: {df.count()}")

Rows after merging: 14776615


#### Filtering trips and showing results

In [ ]:
# PICKUP DATETIME FILTER
rows_before_date = df.count()

# Keep only trips between January 1 and January 7, 2013
df = df.filter(
    (F.col("pickup_datetime") >= "2013-01-01") &
    (F.col("pickup_datetime") <  "2013-01-08")
)

rows_after_date = df.count()
print(f"Rows removed after pickup_datetime filtering: {rows_before_date - rows_after_date}")


# TOTAL AMOUNT FILTER
rows_before_amount = df.count()

# Keep only trips with total_amount > 0, the ones with total_amount equal to or less than 0 
# are likely to be wrong.
df = df.filter(F.col("total_amount") > 0)

rows_after_amount = df.count()
print(f"Rows removed by total_amount filtering: {rows_before_amount - rows_after_amount}")


# TRIP DISTANCE FILTER
rows_before_dist = df.count()

# Keep only trips with trip_distance > 0, the ones with trip_distance equal to or less than 0
# are likely to be wrong.
df = df.filter(F.col("trip_distance") > 0)

rows_after_dist = df.count()
print(f"Rows removed by trip_distance filtering: {rows_before_dist - rows_after_dist}")


# FINAL RESULTS
avg_tip = df.agg(F.avg("tip_amount")).collect()[0][0]

print(f"Total remainingrows after all filters: {df.count()}")
print(f"Average tip amount: ${avg_tip:.2f}")

Rows removed after date filtering: 11766480


Rows removed by total_amount filter: 0


Rows removed by trip_distance filter: 19645


Total rows after all filters: 2990490
Average tip amount: $1.13


### Exercise 2 (12 pts)
For each hour of the day (0–23), compute the average trip duration in minutes and the average trip distance. Provide a graphical representation that allows comparing both metrics across hours side by side. You may want to have a look at: https://docs.bokeh.org/en/latest/docs/user_guide/basic/bars.html#grouping

### Solution 2

#### Calculating the average duration and distance for each hour

In [6]:
# Compute avg trip duration (minutes) and avg distance per hour
hourly_stats = df.groupBy(F.hour("pickup_datetime").alias("hour")) \
    .agg(
        F.avg(F.col("trip_time_in_secs") / 60).alias("avg_duration_min"),
        F.avg("trip_distance").alias("avg_distance")
    ) \
    .orderBy("hour")

hourly_stats.show(24)

26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:03:21 WARN RowBasedKeyValueBatch: Calling spill() on

+----+------------------+------------------+
|hour|  avg_duration_min|      avg_distance|
+----+------------------+------------------+
|   0|11.223678905155799|3.1824580652573524|
|   1|11.414620295666214| 3.187834960243697|
|   2|11.304379693818616| 3.266443770460237|
|   3|11.261855373762803| 3.523590795382152|
|   4|11.762098527361886| 4.130701579528958|
|   5|12.035606561144466| 4.981900548677145|
|   6|10.405564935503474| 3.765743567231717|
|   7|10.550317880700412| 3.146331414658111|
|   8|11.212508254337276|  2.77326200494442|
|   9|10.773916100997631|2.6097442147485355|
|  10|10.651816069479327| 2.713621435157219|
|  11|10.621733516892398| 2.640079624944189|
|  12|10.682610332471548|2.6117908558987577|
|  13| 11.18953992443917|2.7550136559767515|
|  14|11.869509862600193|2.9462623607785976|
|  15|11.826220664083433|2.9449250072488575|
|  16| 11.36870395998212|2.8806185039426806|
|  17|11.609118601008788|2.7296197294264157|
|  18|11.264193100947535| 2.587695134895363|
|  19|10.5

#### Visualizing the average values

In [8]:
output_notebook()

# Prepare data for plotting from the hourly_stats DataFrame, which contains the average trip 
# duration and distance for each hour of the day.
stats = hourly_stats.collect()
hours = [str(row["hour"]) for row in stats]
avg_duration = [row["avg_duration_min"] for row in stats]
avg_distance = [row["avg_distance"] for row in stats]

x = [(h, metric) for h in hours for metric in ["Duration (min)", "Distance (miles)"]]

counts = []
for i in range(len(hours)):
    counts.append(avg_duration[i])
    counts.append(avg_distance[i])

source = ColumnDataSource(dict(x=x, counts=counts))

# Create a grouped bar chart with Bokeh
p = figure(
    x_range=FactorRange(*x),
    height=500,
    width=1200,
    title="Average Trip Duration and Distance by Hour of Day (Jan 1–7, 2013)",
    toolbar_location=None
)

# Make title bigger and bold
p.title.text_font_size = "16px"
p.title.text_font_style = "bold"
p.title.align = "center"

palette = ["steelblue", "orange"]
factors = ["Duration (min)", "Distance (miles)"]

r = p.vbar(
    x="x",
    top="counts",
    width=0.9,
    source=source,
    fill_color=factor_cmap("x", palette=palette, factors=factors, start=1, end=2)
)

# Legend top-right, outside the bars
legend = Legend(items=[
    LegendItem(label="Duration (min)",   renderers=[r], index=0),
    LegendItem(label="Distance (miles)", renderers=[r], index=1),
], location="top_right")


# Move legend outside the plot area (top)
p.add_layout(legend)

# Useful for not overlapping the legend with the bars
p.y_range.start = 0
p.y_range.end   = max(avg_duration) * 1.15

# Remove x-axis tick labels (no more rotated text under bars)
p.xaxis.major_label_text_font_size = "0pt"
p.xgrid.grid_line_color = None
p.yaxis.axis_label = "Value"
p.xaxis.axis_label = "Hour of the Day"
p.xaxis.major_tick_line_color = None  # remove individual bar ticks
p.xaxis.minor_tick_line_color = None  # remove minor ticks
p.xaxis.group_label_orientation = 0
p.xaxis.group_text_font_size = "11px"

show(p)

Loading BokehJS ...

### Exercise 3 (14 pts)
Consider only the boroughs Queens, Staten Island, and EWR. Create a dataframe that shows, for each payment type, the total fare amount collected for trips *originating from* each of those three boroughs, broken down by *destination borough* (including all boroughs as destinations).

> For example, for Queens you should consider:
> - Queens → Queens (cash), Queens → Queens (card), ...
> - Queens → Manhattan (cash), Queens → Manhattan (card), ...
> - and so on for all destination boroughs.


### Solution 3

#### Loading the NYC Boroughs GeoJSON

In [9]:
# Load GeoJSON
with open("data-assignment3/nyc-boroughs.geojson", "r") as f:
    geojson = json.load(f)

borough_shapes = [
    (feature['properties']['borough'], shape(feature['geometry']))
    for feature in geojson['features']
]

# Broadcast to all workers
shapes_broadcast = spark.sparkContext.broadcast(borough_shapes)

#### Defining a function for retrieving the borough based on the coordinates

In [ ]:
# Function to determine the borough based on longitude and latitude
def get_borough(lon, lat):
    if lon is None or lat is None:
        return None # Missing coordinates, cannot determine borough
    point = Point(lon, lat)
    for name, geom in shapes_broadcast.value:
        if geom.contains(point):
            return name
    return None # Points outside all polygons

# Custom operation to get the borough from longitude and latitude
get_borough_udf = F.udf(get_borough, StringType())

#### Adding names of the boroughs to the original dataframe

In [14]:
# Updating the original DataFrame with pickup and dropoff boroughs
df_boroughs = df.withColumn(
    "pickup_borough",
    get_borough_udf(F.col("pickup_longitude"), F.col("pickup_latitude"))
).withColumn(
    "dropoff_borough",
    get_borough_udf(F.col("dropoff_longitude"), F.col("dropoff_latitude"))
).cache()

#### Collecting results for trips with pickups in specific boroughs

In [16]:
# List of boroughs to keep as pickup locations
selected_boroughs = ['Queens', 'Staten Island', 'EWR']

# Dataframe with total fare amount by pickup_borough, dropoff_borough, and payment_type for selected pickup boroughs
# There is a filter to keep only rows where dropoff_borough is not null, to avoid including trips with missing dropoff 
# locations/boroughs in the analysis.
df_result = df_boroughs.filter(F.col("pickup_borough").isin(selected_boroughs)) \
    .filter(F.col("dropoff_borough").isNotNull()) \
    .groupBy("pickup_borough", "dropoff_borough", "payment_type") \
    .agg(F.round(F.sum("fare_amount"), 2).alias("total_fare_amount")) \
    .orderBy("pickup_borough", "dropoff_borough", "payment_type")

df_result.show(df_result.count(), truncate=False)

26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_25 in memory! (computed 1754.3 KiB so far)
26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_26 in memory! (computed 1748.0 KiB so far)
26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_34 in memory! (computed 1750.4 KiB so far)
26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_49 in memory! (computed 1751.8 KiB so far)
26/05/26 10:22:14 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_296_53 in memory.
26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_53 in memory! (computed 384.0 B so far)
26/05/26 10:22:14 WARN MemoryStore: Not enough space to cache rdd_296_54 in memory! (computed 1753.9 KiB so far)
26/05/26 10:22:14 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_296_59 in memory.
26/05/26 10:22:14 WARN MemoryStore: Not enough space to

+--------------+---------------+------------+-----------------+
|pickup_borough|dropoff_borough|payment_type|total_fare_amount|
+--------------+---------------+------------+-----------------+
|Queens        |Bronx          |CRD         |31561.5          |
|Queens        |Bronx          |CSH         |74741.5          |
|Queens        |Bronx          |DIS         |340.0            |
|Queens        |Bronx          |NOC         |573.0            |
|Queens        |Bronx          |UNK         |30.5             |
|Queens        |Brooklyn       |CRD         |465718.5         |
|Queens        |Brooklyn       |CSH         |405903.0         |
|Queens        |Brooklyn       |DIS         |1492.0           |
|Queens        |Brooklyn       |NOC         |1955.0           |
|Queens        |Brooklyn       |UNK         |634.5            |
|Queens        |Manhattan      |CRD         |1918996.52       |
|Queens        |Manhattan      |CSH         |1167401.2        |
|Queens        |Manhattan      |DIS     

### Exercise 4 (16 pts)
Create a dataframe where each row represents a driver, and there is one column per hour of the day (0–23). For each driver-hour, the dataframe provides the maximum number of consecutive trips where the tip amount was strictly greater than $0.

> For example, if for driver B we have trips starting in hour 14 (sorted by pickup time):
>
> - Trip 1: tip = $2.00
> - Trip 2: tip = $0.00
> - Trip 3: tip = $1.50
> - Trip 4: tip = $3.00
>
> The longest streak of tipped trips in hour 14 is 2 (Trips 3 and 4).

Additionally, print the pair (driver, hour) with the maximum streak.

### Solution 4

#### Adding new columns to the dataframe and removing the ones that are not necessary

In [17]:
# Adding column pickup_hour that represents the hour of the day when the trip started, and 
# column tipped that with a boolean value casted to integer (1 if tip_amount > 0, 0 otherwise). 
# I also keep only the relevant columns for the next steps, to optimize the computations.
df_hours_tipped = df.withColumn("pickup_hour", F.hour("pickup_datetime")) \
                .withColumn("tipped", (F.col("tip_amount") > 0).cast("int")) \
                .select("hack_license", "pickup_datetime", "pickup_hour", "tipped")

df_hours_tipped.show(5)

+--------------------+-------------------+-----------+------+
|        hack_license|    pickup_datetime|pickup_hour|tipped|
+--------------------+-------------------+-----------+------+
|39C3E34B3E338A721...|2013-01-02 10:01:42|         10|     1|
|39C3E34B3E338A721...|2013-01-02 12:27:56|         12|     1|
|39C3E34B3E338A721...|2013-01-02 14:09:08|         14|     0|
|39C3E34B3E338A721...|2013-01-02 16:40:21|         16|     0|
|39C3E34B3E338A721...|2013-01-03 14:06:12|         14|     1|
+--------------------+-------------------+-----------+------+
only showing top 5 rows



#### Identifying streaks of consecutive tipped trips

In [ ]:
# Using window functions I can have the trips of each driver grouped by hour, ordered by pickup time, 
# and I can identify consecutive trips with tipped field > 0 (tipped trips).
w = Window.partitionBy("hack_license", "pickup_hour") \
          .orderBy("pickup_datetime") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# row_number is the progressive assigned index of each trip within (driver, hour)
# cumulative_tipped is the cumulative sum of tipped (0/1) up to the current row
# island_id = row_number - cumulative_tipped: for consecutive tipped trips this value stays constant,
# uniquely identifying each streak ("island"). If island_id is null, it means that the trip is not tipped 
# and doesn't belong to any island of tipped trips, if it's not null, it means that the trip is tipped and 
# belongs to the island identified by that id.
# The dataframe is ordered by driver, hour, and pickup time for better readability.
df_islands = df_hours_tipped \
    .withColumn("row_number", F.row_number().over(w)) \
    .withColumn("cumulative_tipped", F.sum("tipped").over(w)) \
    .withColumn(
        "island_id",
        F.when(F.col("tipped") == 1, F.col("row_number") - F.col("cumulative_tipped"))
         .otherwise(None)
    ) \
    .orderBy("hack_license", "pickup_hour", "pickup_datetime")

df_islands.show(20)

+--------------------+-------------------+-----------+------+----------+-----------------+---------+
|        hack_license|    pickup_datetime|pickup_hour|tipped|row_number|cumulative_tipped|island_id|
+--------------------+-------------------+-----------+------+----------+-----------------+---------+
|0002555BBE359440D...|2013-01-02 00:11:39|          0|     0|         1|                0|     NULL|
|0002555BBE359440D...|2013-01-02 00:46:02|          0|     1|         2|                1|        1|
|0002555BBE359440D...|2013-01-03 00:20:55|          0|     0|         3|                1|     NULL|
|0002555BBE359440D...|2013-01-03 00:46:45|          0|     0|         4|                1|     NULL|
|0002555BBE359440D...|2013-01-03 00:57:51|          0|     1|         5|                2|        3|
|0002555BBE359440D...|2013-01-04 00:05:24|          0|     1|         6|                3|        3|
|0002555BBE359440D...|2013-01-04 00:18:51|          0|     1|         7|                4| 

#### Computing the length of each streaks

In [ ]:
# Now, after keeping just the tipped = 1 rows (the others are the ones not tipped and have island_id = null and are useless),
# I can group by hack_license, pickup_hour, and island_id to count the length of each streak of consecutive tipped trips.
# The dataframe is ordered by driver, hour, and streak length for better readability.
streak_df = df_islands \
    .filter(F.col("tipped") == 1) \
    .groupBy("hack_license", "pickup_hour", "island_id") \
    .agg(F.count("*").alias("streak_len")) \
    .select("hack_license", "pickup_hour", "streak_len") \
    .orderBy("hack_license", "pickup_hour", "streak_len")

# In this way I can visualize for each driver, the length of all the streaks (islands) of consecutive 
# tipped trips for each hour of the day.
streak_df.show(20)

26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:51:08 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------+-----------+----------+
|        hack_license|pickup_hour|streak_len|
+--------------------+-----------+----------+
|0002555BBE359440D...|          0|         1|
|0002555BBE359440D...|          0|         3|
|0002555BBE359440D...|          0|         4|
|0002555BBE359440D...|          1|         1|
|0002555BBE359440D...|          1|         1|
|0002555BBE359440D...|          1|         1|
|0002555BBE359440D...|          2|         1|
|0002555BBE359440D...|          2|         1|
|0002555BBE359440D...|          2|         2|
|0002555BBE359440D...|          3|         1|
|0002555BBE359440D...|          3|         2|
|0002555BBE359440D...|          4|         2|
|0002555BBE359440D...|         18|         1|
|0002555BBE359440D...|         18|         1|
|0002555BBE359440D...|         19|         1|
|0002555BBE359440D...|         19|         1|
|0002555BBE359440D...|         19|         2|
|0002555BBE359440D...|         20|         1|
|0002555BBE359440D...|         20|

#### Getting the maximum streak for each (Driver, Hour) pair

In [20]:
# For each (driver, hour) pair, I pick the longest streak among all islands.
# In this way I have just one row for each (driver, hour) pair, with the length 
# of the longest streak of tipped trips in that hour. The dataframe is ordered by driver 
# and hour for better readability.
max_streak_df = streak_df \
    .groupBy("hack_license", "pickup_hour") \
    .agg(F.max("streak_len").alias("max_streak")) \
    .orderBy("hack_license", "pickup_hour")

max_streak_df.show(20)

26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:57:06 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------+-----------+----------+
|        hack_license|pickup_hour|max_streak|
+--------------------+-----------+----------+
|0002555BBE359440D...|          0|         4|
|0002555BBE359440D...|          1|         1|
|0002555BBE359440D...|          2|         2|
|0002555BBE359440D...|          3|         2|
|0002555BBE359440D...|          4|         2|
|0002555BBE359440D...|         18|         1|
|0002555BBE359440D...|         19|         2|
|0002555BBE359440D...|         20|         3|
|0002555BBE359440D...|         21|         4|
|0002555BBE359440D...|         22|         2|
|0002555BBE359440D...|         23|         3|
|0008B3E338CE8C337...|          0|         5|
|0008B3E338CE8C337...|          1|         3|
|0008B3E338CE8C337...|          2|         4|
|0008B3E338CE8C337...|          3|         1|
|0008B3E338CE8C337...|          4|         1|
|0008B3E338CE8C337...|         19|         1|
|0008B3E338CE8C337...|         20|         2|
|0008B3E338CE8C337...|         21|

#### Visualizing drivers' maximum streak by hour of day 

In [21]:
# pivot() transforms the values of pickup_hour (0-23) into separate columns.
# For each driver row, it fills each hour-column with the max_streak value.
# fillna(0) fills missing combinations (driver never worked that hour) with 0.
# The columns are renamed to "00:00", "01:00", ..., "23:00" and the dataframe is ordered 
# by driver for better readability.
pivot_df = max_streak_df \
    .groupBy("hack_license") \
    .pivot("pickup_hour", list(range(24))) \
    .agg(F.first("max_streak")) \
    .fillna(0) \
    .orderBy("hack_license")

for h in range(24):
    if str(h) in pivot_df.columns:
        pivot_df = pivot_df.withColumnRenamed(str(h), f"{h:02d}:00")

# The dataframe shows the drivers as rows and the hours of the day as columns, with the values representing the length 
# of the longest streak of tipped trips in that hour for each driver.
pivot_df.show(20, truncate=False)

26/05/26 10:59:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 10:59:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but re

+--------------------------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|hack_license                    |00:00|01:00|02:00|03:00|04:00|05:00|06:00|07:00|08:00|09:00|10:00|11:00|12:00|13:00|14:00|15:00|16:00|17:00|18:00|19:00|20:00|21:00|22:00|23:00|
+--------------------------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|0002555BBE359440D6CEB34B699D3932|4    |1    |2    |2    |2    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |1    |2    |3    |4    |2    |3    |
|0008B3E338CE8C3377E071A4D80D3694|5    |3    |4    |1    |1    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |1    |2    |2    |4    |4    |
|000A4EBF1CEB9C6BD9978D4362493C6E|2    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    |0    

#### Printing the (Driver, Hour) pair with the longest streak

In [25]:
# Order by max_streak descending and take the first row to find
# the single (driver, hour) pair with the highest streak across all data.
best_row = max_streak_df.orderBy(F.desc("max_streak")).limit(1).collect()[0]

print("\n")
print("(Hack License of the Driver, Pickup Hour) = Max Streak of Tipped Trips")
print(f"({best_row['hack_license']}, {best_row['pickup_hour']}) = {best_row['max_streak']}")

26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/05/26 11:05:55 WARN RowBasedKeyValueBatch: Calling spill() on



(Hack License of the Driver, Pickup Hour) = Max Streak of Tipped Trips
(F9E822E2938FCE35E6064EF828DCD004, 20) = 18


# 🐻‍❄️ Polars Exercises (50 pts)

In this section, you will use **Polars** to perform data cleaning, transformation, and analysis on the NYC taxi dataset.

You will work with the merged dataset obtained from:
- `trip_data.csv`
- `trip_fare.csv`

### Exercise 5 (10 pts)

Perform a sequence of data cleaning steps on the dataset:

1. Remove trips where:
   - `trip_distance <= 10` but `fare_amount > 100`.
   - `trip_distance > 100` or `trip_distance <= 0>` miles.

2. Remove trips with:
   - missing timestamps (`pickup_datetime`, `dropoff_datetime`).
   - `dropoff_datetime <= pickup_datetime`.

After each step:
- Report how many rows were removed.

Finally:
- Report the number of remaining rows.
- Check whether duplicate records exist (based on `medallion`, `hack_license`, `pickup_datetime`).

### Solution 5

### Exercise 6 (12 pts)

Analyze temporal patterns in taxi demand:

1. Group the data by `(weekday, hour)` and compute:
   - total number of trips
   - average fare per trip

2. Visualize the results using a **heatmap**

3. Return the top 5 `(weekday, hour)` by average fare.

### Solution 6

### Exercise 7 (12 pts)

Define a *high-value trip* as one satisfying **at least two** of the following conditions:

- `fare_amount` is in the top 10%
- `tip_amount > 50%` of `fare_amount`
- `trip_distance < 2 miles` AND `fare_amount` above the median

Tasks:

1. Extract all high-value trips.
2. Select only the rides longer than 10 miles (in a straight line).
3. Report the total number of such trips.
4. Create a scatterplot:
   - x-axis: `trip_distance`
   - y-axis: `fare_amount`

4. Briefly interpret the observed patterns

### Solution 7

### Exercise 8 (16 pts)

Analyze driver performance using earnings efficiency:

1. For each trip, compute:
   - trip duration in hours.
   - total earnings = `fare_amount + tip_amount`.

2. Filter:
   - only keep durations between `3 minutes and 5 hours`.

3. For each driver (`hack_license`), compute:
   - total earnings.
   - total driving time (in hours).
   - earnings per hour.
   - total number of trips.

4. Select the **top 15% drivers** based on number of trips.

5. Classify trips into:
   - **day** (i.e., `06:00–18:00`).
   - **night** (remaining hours).

6. Compare driver efficiency:
   - Plot the distribution of earnings per hour for `day vs night drivers` (notice that a driver can be both a "day" and "night" driver in case it performed at least one day ride and one night ride).

7. Answer:
   - Which group appears more efficient?
   - Provide a short explanation based on your results.

### Solution 8